In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from src.utils import read_obs, latexify_xlabel

In [64]:
obs = read_obs("/root/capsule/data/filtered_adata/CaH_final_nuclei.2025-12-29.h5ad")

In [65]:
obs["Supertype"] = obs["Supertype"].str.replace("Supertype", "")
obs["Neighborhood"] = obs["Neighborhood"].str.replace("Neighborhood", "")
obs["library"] = obs.index.map(lambda x: x.split("-")[1])

In [66]:
neurons = ['LGE','MGE', 'CGE', 'LSX ', 'OB']
non_neurons = ['Astro-Epen', 'Immune', 'Vascular', 'OPC-Oligo',]
neuron_obs = obs[obs["Neighborhood"].isin(neurons)]
non_neuron_obs = obs[obs['Neighborhood'].isin(non_neurons)]

In [ ]:
mean_cell_props = neuron_obs.groupby("library")["Supertype"].value_counts(normalize = True).unstack("Supertype").mean()
big_neurons = mean_cell_props[mean_cell_props > 0.01].index.tolist()
mean_cell_props = non_neuron_obs.groupby("library")["Supertype"].value_counts(normalize = True).unstack("Supertype").mean()
big_non_neurons = mean_cell_props[mean_cell_props > 0.01].index.tolist()
big_cells = big_neurons + big_non_neurons

In [74]:
cell_bool = obs.groupby("Donor ID")["Supertype"].value_counts().unstack("Supertype").apply(lambda x: np.any(x < 10))
cells_to_remove= cell_bool[cell_bool].index 

/tmp/ipykernel_49586/3558100039.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  cell_bool = obs.groupby("Donor ID")["Supertype"].value_counts().unstack("Supertype").apply(lambda x: np.any(x < 10))


In [45]:
results = pd.read_csv("/data/Supplemental Tables/Supplemental Table 3.csv", index_col = 0)

In [ ]:
rel_results = results[results["Covariate"].isin(["CPS", "CPS_AT8", "CPS_6e10"])]
rel_results["Regressor"] = rel_results["Covariate"].apply({x:latexify_xlabel(x) for x in ["CPS", "CPS_AT8", "CPS_6e10"]}.get)
rel_results = rel_results[~(rel_results["Cell Type"].isin(cells_to_remove)]

/tmp/ipykernel_49586/1176503063.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rel_results["Regressor"] = rel_results["Covariate"].apply({x:latexify_xlabel(x) for x in ["CPS", "CPS_AT8", "CPS_6e10"]}.get)


In [81]:
sig_cells = []
for i in ["CPS", "CPS_AT8", "CPS_6e10"]:
    p_inclusion = rel_results[(rel_results["Regressor"] == latexify_xlabel(i)) & (rel_results["Reference Cell Type"].isin(big_cells))].groupby("Cell Type")["Inclusion probability"].mean()
    sig_cells.extend(p_inclusion[p_inclusion > 0.85].index.tolist())
    print(sig_cells)
sig_cells = list(set(sig_cells))

[]
['Astro_2', 'Micro-PVM_2_3-SEAAD']
['Astro_2', 'Micro-PVM_2_3-SEAAD']


In [93]:
from matplotlib import lines as mlines
with plt.rc_context({"figure.dpi":600, "figure.figsize":(15, 10), "font.size": 24}):
    scatter = sns.barplot(x = "Cell Type", y = "Final Parameter", hue = "Regressor", data = rel_results[~rel_results["Cell Type"].isin(cells_to_remove)].sort_values("Cell Type"), hue_order = [latexify_xlabel(x) for x in ["CPS", "CPS_AT8", "CPS_6e10"]],  palette = ['#276AB3', "#762a83", "#1b7837"],)
    plt.xticks(rotation = 45,ha = "right")
    plt.ylabel(r"$\beta_{CPS}$")
    plt.xlabel("Cell Type")
    xlabels = [tick.get_text() for tick in scatter.get_xticklabels()]
    plt.xticks(range(len(xlabels)), xlabels)
    for tick in scatter.get_xticklabels():
        if (tick.get_text() in sig_cells):
            tick.set_color("Red")
        # elif (tick.get_position()[1] >= 31):
        #     tick.set_color("Purple")
    # Red Square for Significant
    sig_handle = mlines.Line2D([], [],
                               color='red',           # Edge/Marker color
                               marker='s',            # Square marker
                               linestyle='None',      # No connecting line
                               markersize=10,         # Adjust size for legend clarity
                               label='Significant')
    
    # Black Square for Not Significant
    not_sig_handle = mlines.Line2D([], [],
                                   color='black',
                                   marker='s',
                                   linestyle='None',
                                   markersize=10,
                                   label='Not Significant')
    
    handles, labels = scatter.get_legend_handles_labels()
    handles.extend([sig_handle, not_sig_handle])
    labels.extend(['Significant', 'Not significant'])
    plt.legend(handles=handles, labels=labels, title='Legend', loc='upper right')
    # plt.savefig(
    #     '/scratch/Figure3_Pertpy.svg',
    #     bbox_inches='tight',
    # )
    plt.show()